# Debug List Scraper Malang Times Search
List-only dengan Playwright. URL debug: `https://malangtimes.com/search?keyword=kabupaten+malang`. Tidak scrape artikel detail.


In [12]:
from pathlib import Path
import sys
import importlib
import inspect

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'scrapers').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
pd.set_option('display.max_colwidth', 180)
pd.set_option('display.max_rows', 200)

from scrapers.common import cutoff_date, fetch_html, fetch_text, parse_date, normalize_date, is_older_than_cutoff, records_to_df

cutoff = cutoff_date()
print('project:', PROJECT_ROOT)
print('cutoff:', cutoff.date())

import scrapers.malangtimes as scraper
scraper = importlib.reload(scraper)
from playwright.async_api import async_playwright
print('module file:', scraper.__file__)


project: /Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project
cutoff: 2026-02-28
module file: /Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project/scrapers/malangtimes.py


In [13]:
MAX_PAGES = 200
MAX_CLICKS = 40
TITLE_LIMIT = 90


def short_title(value, limit=TITLE_LIMIT):
    text = '' if pd.isna(value) else str(value).strip()
    return text if len(text) <= limit else text[: limit - 3] + '...'


def ensure_debug_columns(df):
    df = df.copy()
    if 'page_num' not in df.columns:
        df['page_num'] = pd.NA
    if 'published_date' not in df.columns:
        df['published_date'] = None
    df['published_dt'] = df['published_date'].apply(parse_date)
    return df


def print_debug_rows(df):
    for _, row in df.iterrows():
        page = row.get('page_num')
        page_text = '---' if pd.isna(page) else f'{int(page):03d}'
        date_text = row.get('published_date')
        date_text = 'None' if pd.isna(date_text) else str(date_text)
        print(f"page={page_text} | date={date_text} | title={short_title(row.get('title'))}")


def audit_list(df):
    df = ensure_debug_columns(df)
    print('total rows:', len(df))
    if len(df) == 0:
        print('No rows found.')
        return df

    print('first page:', df['page_num'].dropna().min() if df['page_num'].notna().any() else None)
    print('last page:', df['page_num'].dropna().max() if df['page_num'].notna().any() else None)
    print('newest date:', df['published_dt'].max())
    print('oldest date:', df['published_dt'].min())
    print('cutoff date:', cutoff)
    print('reached cutoff:', bool((df['published_dt'].dropna() < cutoff).any()))
    print('null parsed dates:', int(df['published_dt'].isna().sum()))

    print('\nrows per month:')
    display(
        df.assign(month=df['published_dt'].dt.to_period('M').astype(str))
        .groupby('month', dropna=False)
        .size()
        .reset_index(name='count')
    )

    print('\nrows per page:')
    display(
        df.groupby('page_num', dropna=False)
        .agg(
            rows=('url', 'count'),
            newest=('published_dt', 'max'),
            oldest=('published_dt', 'min'),
        )
        .reset_index()
        .tail(60)
    )
    return df


In [14]:
playwright = await async_playwright().start()
browser = await playwright.chromium.launch(headless=True)
page = await browser.new_page()


In [15]:
from bs4 import BeautifulSoup
from datetime import datetime
from dateutil.relativedelta import relativedelta
import re

MALANGTIMES_SEARCH_URL = 'https://malangtimes.com/search?keyword=kabupaten+malang'
RELATIVE_DATE_PATTERN = re.compile(
    r'\b(\d+)\s+(menit|jam|hari|minggu|bulan|tahun)\s+(?:yang\s+)?lalu\b',
    flags=re.IGNORECASE,
)


def parse_relative_date(value, now=None):
    if not value:
        return None

    now = now or datetime.now()
    text = str(value).lower()
    match = RELATIVE_DATE_PATTERN.search(text)

    if not match:
        return None

    amount = int(match.group(1))
    unit = match.group(2)

    if unit == 'menit':
        parsed = now - relativedelta(minutes=amount)
    elif unit == 'jam':
        parsed = now - relativedelta(hours=amount)
    elif unit == 'hari':
        parsed = now - relativedelta(days=amount)
    elif unit == 'minggu':
        parsed = now - relativedelta(weeks=amount)
    elif unit == 'bulan':
        parsed = now - relativedelta(months=amount)
    elif unit == 'tahun':
        parsed = now - relativedelta(years=amount)
    else:
        return None

    return parsed.date().isoformat()


def find_relative_date_text(*values):
    for value in values:
        text = str(value or '').strip()
        match = RELATIVE_DATE_PATTERN.search(text)
        if match:
            return match.group(0).strip()
    return None


def strip_relative_date(value):
    text = str(value or '').strip()
    return RELATIVE_DATE_PATTERN.sub('', text).strip()


def candidate_texts_for_link(link_tag):
    texts = [
        link_tag.get_text(' ', strip=True),
    ]

    title_tag = link_tag.select_one('h3')
    if title_tag:
        texts.append(title_tag.get_text(' ', strip=True))

    current = link_tag
    for _ in range(4):
        current = current.parent
        if not current:
            break
        texts.append(current.get_text(' ', strip=True))

    for sibling in list(link_tag.next_siblings)[:4]:
        if hasattr(sibling, 'get_text'):
            texts.append(sibling.get_text(' ', strip=True))
        else:
            texts.append(str(sibling).strip())

    return texts


async def debug_scrape_search_list(page, max_clicks=MAX_CLICKS):
    rows = []
    seen_urls = set()

    await page.goto(MALANGTIMES_SEARCH_URL, wait_until='networkidle')

    for click_num in range(max_clicks + 1):
        soup = BeautifulSoup(await page.content(), 'html.parser')
        batch_page = click_num + 1

        for link_tag in soup.select("a[href*='/baca/']"):
            url = link_tag.get('href')
            if not url or url in seen_urls:
                continue

            seen_urls.add(url)

            title_tag = link_tag.select_one('h3')
            raw_link_text = link_tag.get_text(' ', strip=True)
            title = title_tag.get_text(' ', strip=True) if title_tag else strip_relative_date(raw_link_text)

            candidate_texts = candidate_texts_for_link(link_tag)
            relative_date = find_relative_date_text(*candidate_texts)
            published_date = parse_relative_date(relative_date)

            rows.append({
                'page_num': batch_page,
                'list_page_url': MALANGTIMES_SEARCH_URL,
                'title': title,
                'url': url,
                'published_date': published_date,
                'relative_date': relative_date,
                'raw_list_text': raw_link_text,
            })

        load_more = page.locator('#load-more')
        if await load_more.count() == 0:
            break

        try:
            await load_more.click(timeout=5000)
            await page.wait_for_timeout(2000)
            print(f'Loaded Malang Times search chunk {batch_page}')
        except Exception as error:
            print(f'Load more stopped: {error}')
            break

    return records_to_df(rows)


list_df = await debug_scrape_search_list(page)
list_df = ensure_debug_columns(list_df)
print_debug_rows(list_df)

print('\nRows with missing relative_date:')
display(
    list_df[list_df['relative_date'].isna()][
        ['page_num', 'title', 'raw_list_text', 'url']
    ].head(20)
)


Loaded Malang Times search chunk 1
Loaded Malang Times search chunk 2
Loaded Malang Times search chunk 3
Loaded Malang Times search chunk 4
Loaded Malang Times search chunk 5
Loaded Malang Times search chunk 6
Loaded Malang Times search chunk 7
Loaded Malang Times search chunk 8
Loaded Malang Times search chunk 9
Loaded Malang Times search chunk 10
Loaded Malang Times search chunk 11
Loaded Malang Times search chunk 12
Loaded Malang Times search chunk 13
Loaded Malang Times search chunk 14
Loaded Malang Times search chunk 15
Loaded Malang Times search chunk 16
Loaded Malang Times search chunk 17
Loaded Malang Times search chunk 18
Loaded Malang Times search chunk 19
Loaded Malang Times search chunk 20
Loaded Malang Times search chunk 21
Loaded Malang Times search chunk 22
Loaded Malang Times search chunk 23
Loaded Malang Times search chunk 24
Loaded Malang Times search chunk 25
Loaded Malang Times search chunk 26
Loaded Malang Times search chunk 27
Loaded Malang Times search chunk 28
L

,page_num,title,raw_list_text,url
7,1,"Ramai Pedagang Pisang Ngeluh Rugi Gegara MBG Libur, Netizen Malah Kompak Bersyukur","2 Ramai Pedagang Pisang Ngeluh Rugi Gegara MBG Libur, Netizen Malah Kompak Bersyukur",https://malangtimes.com/baca/3331346156/20260628/063300/ramai-pedagang-pisang-ngeluh-rugi-gegara-mbg-libur-netizen-malah-kompak-bersyukur
8,1,"Ramai, Curhatan Nadhif Basalamah Alami Pelecehan Seksual di Media Sosial","3 Ramai, Curhatan Nadhif Basalamah Alami Pelecehan Seksual di Media Sosial",https://malangtimes.com/baca/3331346149/20260628/052700/ramai-curhatan-nadhif-basalamah-alami-pelecehan-seksual-di-media-sosial
9,1,Disnaker Kota Malang Siapkan Jalur PHI Soal Kisruh RSI Unisma,4 Disnaker Kota Malang Siapkan Jalur PHI Soal Kisruh RSI Unisma,https://malangtimes.com/baca/3331346147/20260628/043500/disnaker-kota-malang-siapkan-jalur-phi-soal-kisruh-rsi-unisma
10,1,"Viral Polemik Siswa Kota Batu Gagal OSN, Dinas Pendidikan Bantah Lalai dan Sebut Berkas Sudah Lengkap","5 Viral Polemik Siswa Kota Batu Gagal OSN, Dinas Pendidikan Bantah Lalai dan Sebut Berkas Sudah Lengkap",https://malangtimes.com/baca/3331346132/20260628/125700/viral-polemik-siswa-kota-batu-gagal-osn-dinas-pendidikan-bantah-lalai-dan-sebut-berkas-sudah-lengkap
11,1,"Gempa Pacitan M 5,3 Terasa hingga Yogyakarta, BMKG Ungkap Penyebabnya","6 Gempa Pacitan M 5,3 Terasa hingga Yogyakarta, BMKG Ungkap Penyebabnya",https://malangtimes.com/baca/3331346100/20260627/064000/gempa-pacitan-m-5-3-terasa-hingga-yogyakarta-bmkg-ungkap-penyebabnya
12,1,"Korban Latsarmil Kopdes Merah Putih Bertambah: Sudah 5 Nyawa Melayang, Program Masih Berlanjut?","7 Korban Latsarmil Kopdes Merah Putih Bertambah: Sudah 5 Nyawa Melayang, Program Masih Berlanjut?",https://malangtimes.com/baca/3331346081/20260627/012900/korban-latsarmil-kopdes-merah-putih-bertambah-sudah-5-nyawa-melayang-program-masih-berlanjut


In [16]:
list_df = audit_list(list_df)


total rows: 293
first page: 1
last page: 41
newest date: 2026-06-29 00:00:00
oldest date: 2026-03-29 00:00:00
cutoff date: 2026-02-28 09:20:36.028530
reached cutoff: False
null parsed dates: 6

rows per month:


,month,count
0,2026-03,23
1,2026-04,118
2,2026-05,97
3,2026-06,49
4,NaN,6



rows per page:


,page_num,rows,newest,oldest
0,1,13,2026-06-29,2026-06-25
1,2,7,2026-06-24,2026-06-22
2,3,7,2026-06-22,2026-06-22
3,4,7,2026-06-22,2026-06-15
4,5,7,2026-06-15,2026-06-15
5,6,7,2026-06-08,2026-06-08
6,7,7,2026-06-08,2026-06-01
7,8,7,2026-05-29,2026-05-29
8,9,7,2026-05-29,2026-05-29
9,10,7,2026-05-29,2026-05-29


In [17]:
await browser.close()
await playwright.stop()


In [18]:
output_path = PROJECT_ROOT / 'csv' / 'malangtimes_list_debug.csv'
list_df.to_csv(output_path, index=False)
output_path


PosixPath('/Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project/csv/malangtimes_list_debug.csv')

## Optional article sample check

Ambil beberapa artikel detail, cek `content`, lalu simpan ke article debug CSV.

In [ ]:

ARTICLE_SAMPLE_SIZE = 5
ARTICLE_DEBUG_OUTPUT_PATH = PROJECT_ROOT / 'csv' / 'malangtimes_article_debug.csv'

article_rows = []
sample_rows = list_df[list_df['published_date'].notna()].head(ARTICLE_SAMPLE_SIZE)
for index, row in sample_rows.iterrows():
    try:
        article = await scraper.extract_article(page, row)
        article_rows.append(article)
        print(f"[{len(article_rows)}] content_len={len(article.get('content') or '')} | {short_title(article.get('title'))}")
    except Exception as error:
        print(f"failed sample article {index + 1}: {row.get('url')} | {error}")

article_df = pd.DataFrame(article_rows)
ARTICLE_DEBUG_OUTPUT_PATH.parent.mkdir(exist_ok=True)
article_df.to_csv(ARTICLE_DEBUG_OUTPUT_PATH, index=False)
print('saved:', ARTICLE_DEBUG_OUTPUT_PATH)
display(article_df[['title', 'published_date', 'content', 'url']].head() if len(article_df) else article_df)
